In [2]:
import pandas as pd
import re
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)


In [12]:
df_avisos = pd.read_csv("./dados_ondas_calor.csv")
df_sih = pd.read_csv("./dados_sih_completos.csv")

In [14]:
df_avisos['municipio'] = df_avisos['codigo_ibge_municipio'].astype(str).str[:6]
df_sih['municipio'] = df_sih['MUNIC_RES'].astype(str).str[:6]

In [17]:
df_avisos['data'] = pd.to_datetime(df_avisos['data_envio']).dt.tz_localize(None).dt.normalize()
df_sih['data'] = pd.to_datetime(df_sih['DT_INTER'], format='%Y%m%d').dt.tz_localize(None).dt.normalize()


In [19]:
df_avisos = df_avisos.query("data >= '2022-01-01' and data <= '2024-12-31'")
df_sih = df_sih.query("data >= '2022-01-01' and data <= '2024-12-31'")

In [21]:
len(df_sih)

3670832

### Agrupando municipio e dia
- Sempre priorizando a severidade do dia conforme ordem (Extremo, Severo, Moderado)

In [ ]:
severidade_prioridade = {
    'Extreme': 1,
    'Severe': 2,
    'Moderate': 3,
}

df_avisos['prioridade'] = df_avisos['severidade'].map(severidade_prioridade)

avisos_prioritario = df_avisos.loc[
    df_avisos.groupby(['municipio', 'data'])['prioridade'].idxmin()
].copy()

mortes_agrupadas = df_sih.groupby(['municipio', 'data']).size().reset_index(name='qtd_internacoes')

df_full_1 = pd.merge(
    avisos_prioritario[['municipio', 'data', 'severidade']],  
    mortes_agrupadas,
    on=['municipio', 'data'],
    how='outer'
).fillna({'severidade': 'Sem aviso', 'qtd_internacoes': 0})

df_full_1['teve_aviso'] = (df_full_1['severidade'] != 'Sem aviso').astype(int)
df_full_1['data'] = pd.to_datetime(df_full_1['data'])


### Para cada municipio vamos adicionar todos os dias do ano de 2022 até 2024 (mesmo sem aviso ou óbito)

In [23]:
data_inicio = df_full_1['data'].min()
data_fim = df_full_1['data'].max()
datas_completas = pd.date_range(start=data_inicio, end=data_fim, freq='D')
municipios = df_full_1['municipio'].unique()

lista_dfs = []
for mun in municipios:
    df_mun = df_full_1[df_full_1['municipio'] == mun].set_index('data')
    df_mun = df_mun.reindex(datas_completas)
    df_mun['municipio'] = mun
    df_mun['data'] = df_mun.index
    lista_dfs.append(df_mun.reset_index(drop=True))

df_completo = pd.concat(lista_dfs, ignore_index=True)

In [26]:
df_completo['severidade'] = df_completo['severidade'].fillna('Sem aviso')
df_completo['teve_aviso'] = (df_completo['severidade'] != 'Sem aviso').astype(int)
df_completo['qtd_internacoes'] = df_completo['qtd_internacoes'].fillna(0).astype(int)

### Criar coluna de lags 1, 2 e 3 dias

In [27]:
lags = [1, 2, 3]
cols_lag = ['teve_aviso']

df_completo = df_completo.sort_values(['municipio', 'data']).reset_index(drop=True)

for col in cols_lag:
    for lag in lags:
        df_completo[f'{col}_lag_{lag}'] = df_completo.groupby('municipio')[col].shift(lag).fillna(0).astype(int)

df_completo['aviso_lag_0_a_3'] = df_completo[ ['teve_aviso'] + [f'teve_aviso_lag_{l}' for l in lags] ].max(axis=1).astype(int)

### Adicionar dados de regiões de saúde

In [28]:
df_regioes_saude = pd.read_csv("./dados/macroregiao_de_saude.csv", sep=";")
df_regioes_saude = df_regioes_saude[['sg_uf', 'co_uf', 'cod_macrorregiao_de_saude', 'regiao_de_saude', 'macrorregiao_de_saude', 'cod_municipio']]
df_regioes_saude.columns = ['sg_uf', 'co_uf', 'cod_macrorregiao_de_saude', 'regiao_de_saude', 'macrorregiao_de_saude', 'municipio']

In [29]:
df_completo['municipio'] = df_completo['municipio'].astype(int)
df_regioes_saude['municipio'] = df_regioes_saude['municipio'].astype(int)


df_full = pd.merge(
    df_completo,
    df_regioes_saude,
    on=['municipio'],
    how='inner'
)

In [30]:
len(df_full)

6104720

In [31]:
df_full['aviso_lag_0_a_3'] = df_full[ ['teve_aviso'] + [f'teve_aviso_lag_{l}' for l in lags] ].max(axis=1).astype(int)

In [33]:
media_com_aviso = round(df_full[df_full['aviso_lag_0_a_3'] == 1]['qtd_internacoes'].mean(),2)
media_sem_aviso = round(df_full[df_full['aviso_lag_0_a_3'] == 0]['qtd_internacoes'].mean(),2)

RR = media_com_aviso / media_sem_aviso
aumento_percentual = (RR - 1) * 100

print(f"Risco relativo (RR): {RR:.2f}")
print(f"Aumento percentual do risco em dias com aviso: {aumento_percentual:.1f}%")

Risco relativo (RR): 1.23
Aumento percentual do risco em dias com aviso: 23.3%


In [60]:
#110001
#2023-08-22	

In [34]:
df_full['sev_extreme'] = (df_full['severidade'] == 'Extreme').astype(int)
df_full['sev_severe'] = (df_full['severidade'] == 'Severe').astype(int)
df_full['sev_moderate'] = (df_full['severidade'] == 'Moderate').astype(int)
df_full['sev_sem_aviso'] = (df_full['severidade'] == 'Sem aviso').astype(int)  

In [35]:
lags = [1, 2, 3]
for sev in ['sev_extreme', 'sev_severe', 'sev_moderate', 'sev_sem_aviso']:
    for lag in lags:
        df_full[f'{sev}_lag_{lag}'] = df_full.groupby('municipio')[sev].shift(lag).fillna(0).astype(int)


In [36]:
for sev in ['sev_extreme', 'sev_severe', 'sev_moderate', 'sev_sem_aviso']:
    cols_lag = [sev] + [f'{sev}_lag_{lag}' for lag in lags]
    df_full[f'{sev}_lag_0_a_3'] = df_full[cols_lag].max(axis=1).astype(int)


### Indicadores

In [38]:
print("Média mortes dias com qualquer aviso (0 a 3 dias):", 
      round(df_full[df_full['aviso_lag_0_a_3'] == 1]['qtd_internacoes'].mean(),2))
print("Média mortes dias sem aviso (0 a 3 dias):", 
      round(df_full[df_full['aviso_lag_0_a_3'] == 0]['qtd_internacoes'].mean(),2))
print("Média mortes dias com aviso extremo (0 a 3 dias):", 
      round(df_full[df_full['sev_extreme_lag_0_a_3'] == 1]['qtd_internacoes'].mean(),2))
print("Média mortes dias com aviso severo (0 a 3 dias):", 
      round(df_full[df_full['sev_severe_lag_0_a_3'] == 1]['qtd_internacoes'].mean(),2))
print("Média mortes dias com aviso moderado (0 a 3 dias):", 
      round(df_full[df_full['sev_moderate_lag_0_a_3'] == 1]['qtd_internacoes'].mean(),2))

Média mortes dias com qualquer aviso (0 a 3 dias): 0.74
Média mortes dias sem aviso (0 a 3 dias): 0.6
Média mortes dias com aviso extremo (0 a 3 dias): 0.8
Média mortes dias com aviso severo (0 a 3 dias): 0.68
Média mortes dias com aviso moderado (0 a 3 dias): 0.79


In [39]:
def get_severidade_prioritaria(row):
    if row['sev_extreme_lag_0_a_3'] == 1:
        return 'Extreme'
    elif row['sev_severe_lag_0_a_3'] == 1:
        return 'Severe'
    elif row['sev_moderate_lag_0_a_3'] == 1:
        return 'Moderate'
    else:
        return 'Sem aviso'

df_full['severidade_lag_0_a_3'] = df_full.apply(get_severidade_prioritaria, axis=1)


In [66]:
df_full.columns

Index(['municipio', 'severidade', 'qtd_mortes', 'teve_aviso', 'data',
       'teve_aviso_lag_1', 'teve_aviso_lag_2', 'teve_aviso_lag_3',
       'aviso_lag_0_a_3', 'sg_uf', 'co_uf', 'cod_macrorregiao_de_saude',
       'regiao_de_saude', 'macrorregiao_de_saude', 'sev_extreme', 'sev_severe',
       'sev_moderate', 'sev_sem_aviso', 'sev_extreme_lag_1',
       'sev_extreme_lag_2', 'sev_extreme_lag_3', 'sev_severe_lag_1',
       'sev_severe_lag_2', 'sev_severe_lag_3', 'sev_moderate_lag_1',
       'sev_moderate_lag_2', 'sev_moderate_lag_3', 'sev_sem_aviso_lag_1',
       'sev_sem_aviso_lag_2', 'sev_sem_aviso_lag_3', 'sev_extreme_lag_0_a_3',
       'sev_severe_lag_0_a_3', 'sev_moderate_lag_0_a_3',
       'sev_sem_aviso_lag_0_a_3', 'severidade_lag_0_a_3'],
      dtype='object')

In [40]:
df_full=df_full[['municipio', 'severidade', 'qtd_internacoes', 'teve_aviso', 'data','aviso_lag_0_a_3', 'sg_uf', 'co_uf', 'cod_macrorregiao_de_saude','regiao_de_saude', 'macrorregiao_de_saude', 'sev_extreme_lag_0_a_3','sev_severe_lag_0_a_3', 'sev_moderate_lag_0_a_3', 'sev_sem_aviso_lag_0_a_3', 'teve_aviso_lag_3']]

In [41]:
formula = 'qtd_internacoes ~ teve_aviso_lag_3'
import statsmodels.formula.api as smf
modelo = smf.mixedlm(formula, df_full, groups=df_full['cod_macrorregiao_de_saude'])
resultado = modelo.fit()
print(resultado.summary())


            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: qtd_internacoes
No. Observations: 6104720 Method:             REML           
No. Groups:       120     Scale:              5.1714         
Min. group size:  1096    Log-Likelihood:     -13678510.5754 
Max. group size:  161112  Converged:          Yes            
Mean group size:  50872.7                                    
-------------------------------------------------------------
                    Coef.  Std.Err.   z   P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept            2.351    0.793 2.964 0.003  0.796  3.906
teve_aviso_lag_3     0.022    0.009 2.561 0.010  0.005  0.039
Group Var           75.496    1.944                          



In [42]:
formula = 'qtd_internacoes ~ sev_extreme_lag_0_a_3 + sev_severe_lag_0_a_3 + sev_moderate_lag_0_a_3'
import statsmodels.formula.api as smf
modelo = smf.mixedlm(formula, df_full, groups=df_full['cod_macrorregiao_de_saude'])
resultado = modelo.fit()
print(resultado.summary())


              Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  qtd_internacoes
No. Observations:   6104720  Method:              REML           
No. Groups:         120      Scale:               5.1713         
Min. group size:    1096     Log-Likelihood:      -13678428.7849 
Max. group size:    161112   Converged:           Yes            
Mean group size:    50872.7                                      
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               2.349    0.793  2.962 0.003  0.795  3.904
sev_extreme_lag_0_a_3   0.063    0.008  7.687 0.000  0.047  0.079
sev_severe_lag_0_a_3    0.021    0.008  2.728 0.006  0.006  0.036
sev_moderate_lag_0_a_3  0.088    0.009 10.161 0.000  0.071  0.105
Group Var              75.494    1.944                           

